# Forecasting with Otter

This notebook goes from nothing to a 10-day Otter weather forecast. It downloads
a pretrained model from the Hugging Face Hub, forecasts from ERA5 initial
conditions read from the public WeatherBench2 store, scores the forecast
against ERA5, and plots it.

You need a machine with a CUDA GPU and [uv](https://docs.astral.sh/uv/). Set up
the repo and start Jupyter from its root:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh  # If you don't have uv.
git clone https://github.com/cambridge-mlg/otter && cd otter
uv sync
uv run --with jupyter jupyter lab examples/forecast.ipynb
```

No ERA5 download is needed: the initial conditions stream from Google Cloud
Storage.

## Choose a model and an initial time

Each Hub repo holds one model: its weights, its config and a model card.

- `jonasscholz/otter`: deterministic Otter on the 1.5 degree (240 x 121) grid.
- `jonasscholz/otter-xl`: deterministic Otter-XL on the 1.5 degree grid. It is
  more accurate than Otter, and slower.
- `jonasscholz/otter-xl-crps`: Otter-XL on the 1.5 degree grid, fine-tuned
  with a CRPS loss. It forecasts a 10-member ensemble, in about 4 minutes on
  one GPU.
- `jonasscholz/otter-xl-0.25deg`: deterministic Otter-XL on the 0.25 degree
  (1440 x 721) grid. It needs a GPU with plenty of memory: 11 GB is not enough.

In [ ]:
REPO_ID = "jonasscholz/otter"
INIT_TIME = "2020-07-01T00"  # ERA5 is available from 1959 to 2022.
OUTPUT = "forecast.zarr"

## Forecast

`forecast` downloads the model, reads the initial conditions, rolls the model
out for 10 days and writes the forecast as a zarr store following the
WeatherBench2 conventions. Here we save only the headline variables, once a
day. Probabilistic models forecast an ensemble, with the members along a
`realization` dimension.

In [ ]:
from otter.hub import forecast

forecast(
    REPO_ID,
    init_time=INIT_TIME,
    output_path=OUTPUT,
    lead_time_hours=240,
    headline_only=True,
    lead_frequency_hours=24,
)

In [ ]:
import xarray as xr

prediction = xr.open_zarr(OUTPUT).isel(time=0)
prediction

## Score against ERA5

The latitude-weighted root mean squared error of each headline field, as in
WeatherBench2. For an ensemble, we score its mean.

In [ ]:
import numpy as np
import pandas as pd

from otter.hub import WEATHERBENCH2_ERA5

if "realization" in prediction.dims:
    prediction = prediction.mean("realization")
prediction = prediction.load()

grid = (prediction.sizes["latitude"], prediction.sizes["longitude"])
era5 = xr.open_zarr(WEATHERBENCH2_ERA5[grid], storage_options={"token": "anon"})
truth = era5.sel(
    time=prediction.time + prediction.prediction_timedelta,
    latitude=prediction.latitude,
    longitude=prediction.longitude,
)

weights = np.cos(np.deg2rad(prediction.latitude))
weights = weights / weights.mean()
fields = {
    "Z500 (m²/s²)": ("geopotential", 500),
    "T850 (K)": ("temperature", 850),
    "T2M (K)": ("2m_temperature", None),
}
truth_fields = {}
rmse = {}
for name, (variable, level) in fields.items():
    predicted, observed = prediction[variable], truth[variable]
    if level is not None:
        predicted, observed = predicted.sel(level=level), observed.sel(level=level)
    truth_fields[name] = observed = observed.load()
    error = (predicted - observed) ** 2 * weights
    rmse[name] = np.sqrt(error.mean(["latitude", "longitude"])).values

days = prediction.prediction_timedelta.values / np.timedelta64(1, "D")
pd.DataFrame(rmse, index=pd.Index(days.astype(int), name="lead (days)")).round(2)

## Plot

The day-5 500 hPa geopotential forecast, next to ERA5 and the error.

In [ ]:
import matplotlib.pyplot as plt

lead = np.timedelta64(5, "D")
predicted = prediction["geopotential"].sel(level=500, prediction_timedelta=lead)
observed = truth_fields["Z500 (m²/s²)"].sel(prediction_timedelta=lead)

fig, axes = plt.subplots(1, 3, figsize=(18, 4), layout="constrained")
limits = dict(vmin=float(observed.min()), vmax=float(observed.max()))
predicted.plot(ax=axes[0], x="longitude", cmap="viridis", **limits)
axes[0].set_title("Otter, day 5")
observed.plot(ax=axes[1], x="longitude", cmap="viridis", **limits)
axes[1].set_title("ERA5")
(predicted - observed).plot(ax=axes[2], x="longitude", cmap="RdBu_r", center=0)
axes[2].set_title("Otter − ERA5")
for ax in axes:
    ax.set_aspect("equal")
plt.show()

## Next steps

- Forecast from many inits with `end_init_time`, and save all variables by
  dropping `headline_only`.
- The same is available from the command line:
  `uv run python -m otter.hub forecast <repo> --init_time 2020-07-01T00 --output forecast.zarr`.
- `load_pretrained` returns the model and its config, to use the model directly.